# Iterative Content Creator Agent (ADK Capstone)

**Track:** "Concierge Agents" (or "Freestyle")

---

## 1. The Problem

Getting high-quality, refined content from an AI in a single request is difficult. The first draft is often a good start but usually requires multiple follow-up prompts to fix the tone, add details, or correct errors. This manual back-and-forth process is slow and inefficient.

## 2. The Solution

This project solves that problem by building a **multi-agent system** using the Agent Development Kit (ADK) that **automatically iterates and refines content**.

Instead of just one agent, this system is a "pipeline" of specialist agents that work together in a loop:
1.  An `InitialWriterAgent` creates the first draft.
2.  A `CriticAgent` reviews the draft and provides constructive feedback.
3.  A `RefinerAgent` reads the feedback and rewrites the draft to improve it.
4.  This "Critique -> Refine" loop repeats (up to 3 times) until the `CriticAgent` "APPROVES" the content.

This "assembly line" mimics a human writing team and produces a much higher-quality result than a single-shot prompt.



## 3. ADK Concepts Used

This project is built using the core architectural patterns from the "5-Day AI Agents Intensive Course":

* **`LlmAgent` (Day 1a):** The base for all our specialist agents (`InitialWriterAgent`, `CriticAgent`, `RefinerAgent`).
* **`SequentialAgent` (Day 1b):** Used as the main `root_agent` to define the high-level flow: **Step 1 (Write)** -> **Step 2 (Refine Loop)**.
* **`LoopAgent` (Day 1b):** The core of this project. It runs the `CriticAgent` and `RefinerAgent` repeatedly to improve the content.
* **`FunctionTool` (Day 2a):** A custom Python tool (`exit_loop`) is used to give the `RefinerAgent` a way to programmatically stop the `LoopAgent` once the "APPROVED" signal is received.
* **`InMemoryRunner` (Day 1a):** Used to run and test the entire agent pipeline directly inside this notebook.

## 4. How to Use This Notebook

1.  **Run Cell 1:** Installs `google-adk` and sets up the Google API Key.
2.  **Run Cell 2:** Imports all necessary ADK components.
3.  **Run Cell 3:** Defines all the specialist agents, tools, and the final `root_agent` workflow.
4.  **Run Cell 4:** This is the final step! It uses `InMemoryRunner` to start the agent with a prompt. You can see the full, multi-step process (draft, critique, refinement) printed in the output of this cell.

In [1]:
# Install the Agent Development Kit
!pip install google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/319.9 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 9.8 MB/s eta 0:00:00


  Attempting uninstall: protobuf


    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:


      Successfully uninstalled protobuf-6.33.0


  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.1
    Uninstalling cachetools-6.2.1:
      Successfully uninstalled cachetools-6.2.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.10.0 which is incompatible.


In [2]:


import os
from kaggle_secrets import UserSecretsClient
from google.genai import types

# Load the API key from Kaggle Secrets
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ GOOGLE_API_KEY loaded from Kaggle Secrets.")
except Exception as e:
    print("🔑 Please add your 'GOOGLE_API_KEY' to Kaggle Secrets.")
    print("   (Click Add-ons > Secrets > New Secret)")

✅ GOOGLE_API_KEY loaded from Kaggle Secrets.


In [3]:
from google.adk.agents import LlmAgent, SequentialAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner

print("✅ ADK components imported.")

✅ ADK components imported.


In [4]:
from google.adk.agents import LlmAgent, SequentialAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner

print("✅ ADK components imported.")

✅ ADK components imported.


In [5]:
# --- 1. Define The `exit_loop` Tool (from Day 2a) ---
# This is the custom Python function the agent will call to stop.
def exit_loop():
    """
    Call this function ONLY when the critique is 'APPROVED'.
    This signals that the story is finished and the loop should stop.
    """
    print("--- 🔁 Loop: 'APPROVED' received. Exiting loop. ---")
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}

# --- 2. Define the Model ---
# Standard retry config
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)
model = Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config)

# --- 3. Define Specialist Agents (from Day 1b) ---

# Agent 1: Runs ONCE at the start.
initial_writer_agent = LlmAgent(
    model=model,
    name="InitialWriterAgent",
    instruction="""Based on the user's prompt, write the first draft of a short blog post (around 100 words).
    Output only the blog post text, with no introduction or explanation.""",
    output_key="draft", # Stores the output in a variable named "draft"
)

# Agent 2: Runs INSIDE the loop. It provides feedback.
critic_agent = LlmAgent(
    model=model,
    name="CriticAgent",
    instruction="""You are a constructive critic. Review the blog post draft provided below.
    Blog Draft: {draft}

    Evaluate the draft's clarity and engagement.
    - If the draft is excellent, you MUST respond with the exact word: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable bullet points for improvement.""",
    output_key="critique", # Stores the feedback in "critique"
)

# Agent 3: Runs INSIDE the loop. It decides whether to stop or rewrite.
refiner_agent = LlmAgent(
    model=model,
    name="RefinerAgent",
    instruction="""You are a blog post refiner. You have a draft and a critique.
    
    Previous Draft: {draft}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function.
    - OTHERWISE, rewrite the entire blog post draft, fully incorporating the critique.""",
    output_key="draft", # Overwrites the "draft" variable with the new version
    tools=[FunctionTool(exit_loop)], # Gives the agent the "exit" tool
)

# --- 4. Build The Workflow (from Day 1b) ---

# This is the loop: Critique -> Refine
story_refinement_loop = LoopAgent(
    name="RefinementLoop",
    sub_agents=[
        critic_agent,
        refiner_agent,
    ],
    max_iterations=3, # Will only try to refine 3 times max
)

# This is the main sequence: Write Initial Draft -> Start Loop
root_agent = SequentialAgent(
    name="CreatorPipeline",
    sub_agents=[
        initial_writer_agent,
        story_refinement_loop,
    ],
)

print("✅ All agents and workflows are defined.")

✅ All agents and workflows are defined.


In [6]:
# Create the runner and point it to your main agent
runner = InMemoryRunner(agent=root_agent)

# Run the agent!
print("🚀 Starting the Iterative Content Creator Agent...")

# We use await because the agent runs asynchronously
response = await runner.run_debug(
    "Write a short, engaging blog post about the challenges of space exploration."
)

print("\n--- 🏁 WORKFLOW COMPLETE ---")

🚀 Starting the Iterative Content Creator Agent...

 ### Created new session: debug_session_id

User > Write a short, engaging blog post about the challenges of space exploration.


InitialWriterAgent > The allure of the stars is powerful, but space exploration is a monumental undertaking. Reaching orbit alone requires overcoming Earth's immense gravity, a feat demanding incredible power and precision. Once there, the vacuum of space presents its own set of dangers: extreme temperatures, radiation, and micrometeoroids. Long-duration missions introduce physiological challenges, from bone density loss to psychological strain. The sheer cost is also a significant hurdle, requiring vast resources and global cooperation. Yet, despite these daunting obstacles, humanity's drive to explore continues, pushing the boundaries of what's possible and revealing the universe's wonders.


CriticAgent > Here are a few suggestions to enhance the blog post draft:

*   **Strengthen the Opening Hook:** While the current opening is functional, consider a more evocative or question-driven sentence to immediately grab the reader's attention. For instance, "What draws us to the silent, star-dusted expanse above?" or "The night sky whispers promises of the unknown, but reaching it is a Herculean task."
*   **Add a Touch of Sensory Detail or Anecdote:** To make the dangers more palpable, briefly describe *how* these challenges affect astronauts. Instead of just listing "extreme temperatures," you could add a phrase like "where survival hinges on precisely engineered suits against both searing heat and bone-chilling cold." A brief mention of a known challenge faced by astronauts (like the psychological impact of confinement) could also add depth.
*   **Consider a More Forward-Looking Conclusion:** The current conclusion is good, but for a blog post, a slightly more inspiring or tho

RefinerAgent > What draws us to the silent, star-dusted expanse above? The allure of the cosmos is undeniable, but venturing into space is a monumental undertaking. Breaking free from Earth's immense gravity demands incredible power and precision. Once in orbit, the vacuum of space becomes a formidable adversary, with extreme temperatures where survival hinges on precisely engineered suits against both searing heat and bone-chilling cold, relentless radiation, and the constant threat of micrometeoroids. Long-duration missions add further layers of challenge, introducing physiological hurdles like bone density loss and the psychological strain of confinement, a well-documented concern for astronauts. The sheer cost is also a significant hurdle, requiring vast resources and unprecedented global cooperation. Yet, despite these daunting obstacles, humanity's innate drive to explore propels us forward, pushing the boundaries of what's possible. This relentless pursuit not only answers our d

CriticAgent > APPROVED


--- 🔁 Loop: 'APPROVED' received. Exiting loop. ---


RefinerAgent > The blog post has been approved! We're done with the refinement process.


CriticAgent > APPROVED


--- 🔁 Loop: 'APPROVED' received. Exiting loop. ---


RefinerAgent > The blog post has been approved! We're done with the refinement process.

--- 🏁 WORKFLOW COMPLETE ---


In [ ]:
# Creates a new folder named 'my_agent_folder'
!adk create my_agent_folder --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY

In [ ]:
%%writefile /kaggle/working/creator_agent/agent.py

import os
from kaggle_secrets import UserSecretsClient
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool

# --- 1. API Key and Model Setup ---

# Load the API key from Kaggle Secrets
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
except Exception as e:
    # This just means it's already set in the environment
    pass

# Standard retry config
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

# Define the model we'll use for all agents
model = Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config)


# --- 2. Define The `exit_loop` Tool (from Day 2a) ---
def exit_loop():
    """
    Call this function ONLY when the critique is 'APPROVED'.
    This signals that the story is finished and the loop should stop.
    """
    print("--- 🔁 Loop: 'APPROVED' received. Exiting loop. ---")
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


# --- 3. Define Specialist Agents (from Day 1b) ---

# Agent 1: Runs ONCE at the start.
initial_writer_agent = LlmAgent(
    model=model,
    name="InitialWriterAgent",
    instruction="""Based on the user's prompt, write the first draft of a short blog post (around 100 words).
    Output only the blog post text, with no introduction or explanation.""",
    output_key="draft", # Stores the output in a variable named "draft"
)

# Agent 2: Runs INSIDE the loop. It provides feedback.
critic_agent = LlmAgent(
    model=model,
    name="CriticAgent",
    instruction="""You are a constructive critic. Review the blog post draft provided below.
    Blog Draft: {draft}

    Evaluate the draft's clarity and engagement.
    - If the draft is excellent, you MUST respond with the exact word: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable bullet points for improvement.""",
    output_key="critique", # Stores the feedback in "critique"
)

# Agent 3: Runs INSIDE the loop. It decides whether to stop or rewrite.
refiner_agent = LlmAgent(
    model=model,
    name="RefinerAgent",
    instruction="""You are a blog post refiner. You have a draft and a critique.
    
    Previous Draft: {draft}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function.
    - OTHERWISE, rewrite the entire blog post draft, fully incorporating the critique.""",
    output_key="draft", # Overwrites the "draft" variable with the new version
    tools=[FunctionTool(exit_loop)], # Gives the agent the "exit" tool
)


# --- 4. Build The Workflow (from Day 1b) ---

# This is the loop: Critique -> Refine
story_refinement_loop = LoopAgent(
    name="RefinementLoop",
    sub_agents=[
        critic_agent,
        refiner_agent,
    ],
    max_iterations=3, # Will only try to refine 3 times max
)

# This is the main sequence: Write Initial Draft -> Start Loop
# This is the 'root_agent' that ADK will run
root_agent = SequentialAgent(
    name="CreatorPipeline",
    sub_agents=[
        initial_writer_agent,
        story_refinement_loop,
    ],
)

print("✅ 'creator_agent/agent.py' file written successfully.")

In [ ]:
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers

# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    # --- This HTML part is from the notebooks ---
    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """
    display(HTML(styled_html))
    return url_prefix

In [ ]:

url_prefix = get_adk_proxy_url()

In [ ]:

!adk web --url_prefix {url_prefix}